In [15]:
import pandas as pd
import glob
import os

def load_all_logs(log_root_dir):
    all_files = glob.glob(os.path.join(log_root_dir, "**", "*.csv"), recursive=True)
    
    log_dfs = []
    for file in all_files:
        df = pd.read_csv(file)
        log_dfs.append(df)

    logs = pd.concat(log_dfs, ignore_index=True)
    return logs

In [42]:
def label_attack_flows(flow_csv, log_root_dir, output_csv):
    print(f"Labeling {flow_csv} using logs in {log_root_dir}...")

    flows = pd.read_csv(flow_csv)
    logs = load_all_logs(log_root_dir)



    # Convert logs to UTC epoch seconds
    logs["timestamp"] = pd.to_datetime(logs["Timestamp"], utc=True, errors="coerce")
    # print(logs["timestamp"].astype("int64").head())

    logs = logs.dropna(subset=["timestamp"])
    # logs["epoch"] = logs["timestamp"].astype("int64") // 10**9
    logs["epoch"] = logs["timestamp"].apply(lambda x: int(x.timestamp()))
    logs["time_window"] = logs["epoch"].astype(int)

    flows["time_window"] = flows["time_window"].astype(int)

    merged = flows.merge(
        logs[["time_window", "TargetIP", "Attack"]],
        left_on=["time_window", "ip.dst"],
        right_on=["time_window", "TargetIP"],
        how="left"
    )

    # st = set(flows["time_window"]).intersection(set(logs["time_window"]))
    # if (len(st) == 0):
    #     print("bruh")
    # else:
    #     print("BRUH!", end = " ")
    #     print(len(st))

    # flow_pairs = set(zip(flows["time_window"], flows["ip.dst"]))
    # log_pairs = set(zip(logs["time_window"], logs["TargetIP"]))
    
    # print("Pair intersection:", len(flow_pairs.intersection(log_pairs)))
    
    

    merged["is_attack"] = merged["Attack"].notna().astype(int)
    merged["attack_type"] = merged["Attack"].fillna("benign")

    merged.drop(columns=["TargetIP", "Attack"], inplace=True)

    print("Attack rows:", merged["is_attack"].sum())

    # print("Flow time range:")
    # print(flows["time_window"].min(), flows["time_window"].max())

    
    # print("Log time range:")
    # print(logs["epoch"].min(), logs["epoch"].max())
    

    merged.to_csv(output_csv, index=False)
    print(f"Saved → {output_csv}\n")


In [ ]:

# # Label external attack
# label_attack_flows(
#     "../train/external_flows.csv",
#     "../data/attack/external/network-wide/",   # <-- put correct log filename
#     "external_labeled.csv"
# )

In [44]:

label_attack_flows(
    "../train/cscada_flows.csv",
    "../data/attack/compromised-scada/attack logs",     # <-- put correct log filename
    "../train/cscada_labeled.csv"
)

Labeling ../train/cscada_flows.csv using logs in ../data/attack/compromised-scada/attack logs...
Attack rows: 146427
Saved → ../train/cscada_labeled.csv



In [45]:
df = pd.read_csv("../train/cscada_labeled.csv")
print(df["is_attack"].value_counts())


is_attack
0    664681
1    146427
Name: count, dtype: int64
